1. EXTRACTION ET NETTOYAGE DE TEXTE DEPUIS DES PDF

In [7]:
import sys
!{sys.executable} -m pip install pymupdf

  Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl.metadata (24 kB)
Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl (19.2 MB)



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import sys
!{sys.executable} -m pip install pdfplumber

  Using cached pdfplumber-0.11.9-py3-none-any.whl.metadata (43 kB)
  Using cached pdfminer_six-20251230-py3-none-any.whl.metadata (4.3 kB)
  Using cached pypdfium2-5.8.0-py3-none-win_amd64.whl.metadata (68 kB)
  Using cached cryptography-48.0.0-cp311-abi3-win_amd64.whl.metadata (4.3 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached pdfplumber-0.11.9-py3-none-any.whl (60 kB)
Using cached pdfminer_six-20251230-py3-none-any.whl (6.6 MB)
Using cached cryptography-48.0.0-cp311-abi3-win_amd64.whl (3.8 MB)
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import sys
!{sys.executable} -m pip install pytesseract

  Using cached pytesseract-0.3.13-py3-none-any.whl.metadata (11 kB)
Using cached pytesseract-0.3.13-py3-none-any.whl (14 kB)



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import sys
!{sys.executable} -m pip install pdf2image

  Using cached pdf2image-1.17.0-py3-none-any.whl.metadata (6.2 kB)
Using cached pdf2image-1.17.0-py3-none-any.whl (11 kB)



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import sys
!{sys.executable} -m pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os
import re
import fitz                  # PyMuPDF
import pdfplumber
import pytesseract

from pdf2image import convert_from_path
from tqdm import tqdm


2. CONFIGURATION

In [21]:
# Dossier contenant les PDF
INPUT_DIR = r"C:\Users\yefif\AI_Projets\rag-assistant-ai\data\raw"

# Dossier de sortie des fichiers texte
OUTPUT_DIR = r"C:\Users\yefif\AI_Projets\rag-assistant-ai\data\output"

# Langue OCR
LANGUAGE = "fra"




 3. FONCTION DE NETTOYAGE

In [22]:
def clean_text(text):
    """
    Nettoie le texte extrait.
    """

    if not text:
        return ""

    # Supprime les espaces multiples
    text = re.sub(r"\s+", " ", text)

    # Supprime caractères invisibles
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)

    # Supprime les doubles ponctuations
    text = re.sub(r"([.,;:!?]){2,}", r"\1", text)

    # Supprime numéros de page
    text = re.sub(r"\bPage\s+\d+\b", "", text)

    # Recolle les mots coupés
    text = re.sub(r"-\s+", "", text)

    # Supprime espaces multiples
    text = re.sub(r"\s{2,}", " ", text)

    return text.strip()



4. EXTRACTION 

4.1. EXTRACTION STANDARD

In [23]:
def extract_text_pymupdf(pdf_path):
    """
    Extraction rapide avec PyMuPDF.
    """

    text = []

    try:
        doc = fitz.open(pdf_path)

        for page in doc:

            # Extraction texte page
            page_text = page.get_text("text")

            text.append(page_text)

        doc.close()

    except Exception as e:
        print(f"Erreur PyMuPDF : {e}")

    return "\n".join(text)



4.2.EXTRACTION PDF COMPLEXE

In [24]:
def extract_text_pdfplumber(pdf_path):
    """
    Extraction alternative pour PDF complexes.
    """

    text = []

    try:
        with pdfplumber.open(pdf_path) as pdf:

            for page in pdf.pages:

                page_text = page.extract_text()

                if page_text:
                    text.append(page_text)

    except Exception as e:
        print(f"Erreur pdfplumber : {e}")

    return "\n".join(text)



 4.3. OCR POUR PDF SCANNÉS

In [25]:
def extract_text_ocr(pdf_path):
    """
    OCR complet via Tesseract.
    """

    text = []

    try:

        # Conversion PDF -> images
        images = convert_from_path(pdf_path)

        for image in images:

            # OCR
            ocr_text = pytesseract.image_to_string(
                image,
                lang=LANGUAGE
            )

            text.append(ocr_text)

    except Exception as e:
        print(f"Erreur OCR : {e}")

    return "\n".join(text)



4.4 DÉTECTION PDF SCANNÉ

In [26]:
def is_scanned_pdf(pdf_path):
    """
    Vérifie si le PDF contient du texte.
    """

    try:

        doc = fitz.open(pdf_path)

        for page in doc:

            text = page.get_text("text")

            # Si du texte existe
            if text.strip():

                doc.close()

                return False

        doc.close()

    except:
        pass

    return True


4.5. TRAITEMENT D'UN PDF

In [29]:
def process_pdf(pdf_path):

    filename = os.path.basename(pdf_path)

    output_file = os.path.join(
        OUTPUT_DIR,
        filename.replace(".pdf", ".txt")
    )

    print(f"\nTraitement : {filename}")

    text = ""

    # Vérifie si PDF scanné
    scanned = is_scanned_pdf(pdf_path)

    # =====================================================
    # CAS PDF SCANNÉ
    # =====================================================

    if scanned:

        print("-> PDF scanné détecté")

        text = extract_text_ocr(pdf_path)
    
    # CAS PDF TEXTE

    else:

        print("-> PDF texte détecté")

        # Extraction principale
        text = extract_text_pymupdf(pdf_path)

        # Fallback
        if len(text.strip()) < 100:

            print("-> Fallback pdfplumber")

            text = extract_text_pdfplumber(pdf_path)

    # Nettoyage
    cleaned_text = clean_text(text)

    # Sauvegarde
    with open(output_file, "w", encoding="utf-8") as f:

        f.write(cleaned_text)

    print(f"-> Sauvegardé : {output_file}")



5. TRAITEMENT DU CORPUS

In [30]:
def main():

    # Liste des PDF
    pdf_files = [

        os.path.join(INPUT_DIR, f)

        for f in os.listdir(INPUT_DIR)

        if f.lower().endswith(".pdf")
    ]

    print(f"Nombre de PDF détectés : {len(pdf_files)}")

    # Traitement
    for pdf_file in tqdm(pdf_files):

        process_pdf(pdf_file)

    print("\nTraitement terminé.")

6. LANCEMENT

In [31]:
if __name__ == "__main__":

    main()

Nombre de PDF détectés : 5


  0%|          | 0/5 [00:00<?, ?it/s]


Traitement : 1543744461-Manuel_de_proc_dures_RH_de_lONEAD.pdf
-> PDF texte détecté


 20%|██        | 1/5 [00:00<00:01,  3.66it/s]

-> Sauvegardé : C:\Users\yefif\AI_Projets\rag-assistant-ai\data\output\1543744461-Manuel_de_proc_dures_RH_de_lONEAD.txt

Traitement : MANUEL DE PROCEDURE DRH PME.pdf
-> PDF texte détecté


...

-> Sauvegardé : C:\Users\yefif\AI_Projets\rag-assistant-ai\data\output\MANUEL DE PROCEDURES EN RESSOURCES HUMAINES.txt

Traitement : ORGANISATION INTERNATIONALE DU TRAVAIL.pdf
-> PDF texte détecté


 80%|████████  | 4/5 [00:02<00:00,  1.44it/s]

-> Sauvegardé : C:\Users\yefif\AI_Projets\rag-assistant-ai\data\output\ORGANISATION INTERNATIONALE DU TRAVAIL.txt

Traitement : pub-2020001249601F-mca-human-resources-f.pdf
-> PDF texte détecté


100%|██████████| 5/5 [00:03<00:00,  1.45it/s]

-> Sauvegardé : C:\Users\yefif\AI_Projets\rag-assistant-ai\data\output\pub-2020001249601F-mca-human-resources-f.txt

Traitement terminé.
